# IMPORTING LIBRARIES

In [1]:
from flask import Flask, render_template, Response, request
import cv2
import numpy as np
from flask_ngrok import run_with_ngrok
import base64
from PIL import Image
import io

import mlflow
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as F
from torchvision.transforms import Compose, ToTensor, Resize, Normalize

c:\Users\anike\anaconda3\envs\data-science\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = torch.load("saved_models/model.pth", pickle_module=mlflow.pytorch.pickle_module)

# FLASK

In [6]:
app = Flask(__name__)
run_with_ngrok(app)

def predict(image, model):
    image = Image.fromarray(np.array(image)[...,:3])
    image = Compose([Resize((256, 256)), ToTensor(), Normalize(
        mean = [0.485, 0.456, 0.406],
        std = [0.229, 0.224, 0.225])])(image).unsqueeze(0).to("cuda")
    model.eval()
    with torch.no_grad():
        output = model(image)

    output = (output.squeeze().cpu().numpy() > 0.5).astype('float32')
    output = cv2.cvtColor(output, cv2.COLOR_GRAY2BGR)
    output = Image.fromarray((output*255).astype(np.uint8))
    return output
    
@app.route('/')
def index():
    return render_template('index.html')

@app.route('/test', methods=['GET','POST'])
def test():
    if request.method == "POST":
        x = request.data.decode("utf-8")
        filename = 'some_image.jpg'
        with open(filename, 'wb') as f:
            f.write(base64.b64decode(x))
        image = Image.open(filename)
        im = predict(image, model)
        im = im.transpose(Image.FLIP_LEFT_RIGHT)
        data = io.BytesIO()
        im.save(data, "JPEG")
        encoded_img_data = base64.b64encode(data.getvalue())
        return Response(encoded_img_data.decode('utf-8'), mimetype='multipart/x-mixed-replace; boundary=frame')

if __name__=='__main__':
    app.run()

 * Serving Flask app '__main__' (lazy loading)
 * Environment: production
   Use a production WSGI server instead.
 * Debug mode: off


 * Running on http://127.0.0.1:5000/ (Press CTRL+C to quit)
127.0.0.1 - - [16/Feb/2024 00:05:57] "POST /test HTTP/1.1" 200 -
127.0.0.1 - - [16/Feb/2024 00:05:58] "POST /test HTTP/1.1" 200 -
127.0.0.1 - - [16/Feb/2024 00:05:59] "POST /test HTTP/1.1" 200 -
127.0.0.1 - - [16/Feb/2024 00:06:00] "POST /test HTTP/1.1" 200 -
127.0.0.1 - - [16/Feb/2024 00:06:01] "POST /test HTTP/1.1" 200 -
127.0.0.1 - - [16/Feb/2024 00:06:02] "POST /test HTTP/1.1" 200 -
127.0.0.1 - - [16/Feb/2024 00:06:02] "POST /test HTTP/1.1" 200 -
Exception in thread Thread-199:
Traceback (most recent call last):
  File "c:\Users\anike\anaconda3\envs\data-science\lib\site-packages\urllib3\connection.py", line 175, in _new_conn
    (self._dns_host, self.port), self.timeout, **extra_kw
  File "c:\Users\anike\anaconda3\envs\data-science\lib\site-packages\urllib3\util\connection.py", line 95, in create_connection
    raise err
  File "c:\Users\anike\anaconda3\envs\data-science\lib\site-packages\urllib3\util\connection.py", line